# RAG: Document Question Answering Pipeline

A complete **Retrieval-Augmented Generation (RAG)** implementation using LangChain, covering document ingestion through to multi-turn conversational Q&A.

**Pipeline overview:**

```
Documents -> Split -> Embed -> VectorStore
                                    |
User Query -> Retrieve -> LLM -> Answer
```

**Parts covered:**
1. Document Loading
2. Document Splitting
3. Vector Stores and Embeddings
4. Advanced Retrieval
5. Question Answering
6. Conversational Chat with Memory
7. Alternative Stack: FAISS + Nomic Embeddings

**Dependencies (all free):**
- Embeddings: `all-MiniLM-L6-v2` via HuggingFace — runs locally on CPU, no API key required
- LLM: Groq free tier — [create a key here](https://console.groq.com/keys)

## Setup and Installation

In [ ]:
!pip install -q langchain langchain-community langchain-groq \
    chromadb sentence-transformers pypdf beautifulsoup4 lark scikit-learn docarray \
    langchain-huggingface faiss-cpu einops

In [ ]:
import os
from getpass import getpass

# Groq API key — free tier, no credit card required: https://console.groq.com/keys
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")
print("API key set.")

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import shutil
import time

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq

print("Loading embedding model (approx. 90MB, downloaded once)...")
embedding = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

# LLM configuration
# Available models on Groq free tier:
#   "llama-3.1-8b-instant"     -- fastest, lowest latency
#   "llama-3.3-70b-versatile"  -- highest quality responses
#   "mixtral-8x7b-32768"       -- long context window (32k tokens)
#   "gemma2-9b-it"             -- Google Gemma 2

GROQ_MODEL = "llama-3.1-8b-instant"  # change this to switch models

llm = ChatGroq(model=GROQ_MODEL, temperature=0)


def safe_invoke(chain, inputs, delay=5, retries=3):
    """Invoke a LangChain chain with automatic retry on rate-limit errors (HTTP 429)."""
    for attempt in range(1, retries + 1):
        try:
            return chain.invoke(inputs)
        except Exception as e:
            if "429" in str(e) or "rate_limit" in str(e).lower() or "rate limit" in str(e).lower():
                wait = delay * attempt
                print(f"Rate limit hit (attempt {attempt}/{retries}). Retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError(f"Chain invocation failed after {retries} retries.")


print(f"Embedding model and LLM ({GROQ_MODEL}) initialised.")
print("To use a higher-quality model, set GROQ_MODEL = 'llama-3.3-70b-versatile'")

---
## Part 1: Document Loading

Two sources are loaded:
- **Wikipedia articles** (Machine Learning, Deep Learning, NLP, Large Language Models)
- **"Attention Is All You Need"** — the original Transformer paper from arXiv

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

url = "https://en.wikipedia.org/wiki/Machine_learning"
loader = WebBaseLoader(url)
web_docs = loader.load()

print(f"Documents loaded: {len(web_docs)}")
print(f"Total characters: {len(web_docs[0].page_content):,}")
print("\n--- First 500 characters ---")
print(web_docs[0].page_content[:500])
print("Metadata:", web_docs[0].metadata)

In [ ]:
# Load four Wikipedia articles to build a broader knowledge base
urls = [
    "https://en.wikipedia.org/wiki/Machine_learning",
    "https://en.wikipedia.org/wiki/Deep_learning",
    "https://en.wikipedia.org/wiki/Natural_language_processing",
    "https://en.wikipedia.org/wiki/Large_language_model",
]

loader = WebBaseLoader(urls)
all_docs = loader.load()

print(f"Documents loaded: {len(all_docs)}")
for doc in all_docs:
    print(f"  {doc.metadata.get('source', 'unknown')} -- {len(doc.page_content):,} chars")

In [ ]:
import urllib.request
from langchain_community.document_loaders import PyPDFLoader

pdf_url = "https://arxiv.org/pdf/1706.03762"
pdf_path = "/tmp/attention_is_all_you_need.pdf"

print("Downloading Transformer paper PDF...")
urllib.request.urlretrieve(pdf_url, pdf_path)

pdf_loader = PyPDFLoader(pdf_path)
pdf_pages = pdf_loader.load()

print(f"Pages loaded: {len(pdf_pages)}")
print("\n--- Page 1 preview ---")
print(pdf_pages[0].page_content[:600])
print("Page metadata:", pdf_pages[0].metadata)

---
## Part 2: Document Splitting

Documents are split into overlapping chunks before embedding. Two parameters control this:

- `chunk_size`: maximum characters per chunk
- `chunk_overlap`: number of characters shared between adjacent chunks (preserves context at boundaries)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter

r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=20,
    separators=["\n\n", "\n", ". ", " ", ""]
)

sample_text = """Machine learning (ML) is a field of study in artificial intelligence concerned
with the development of statistical algorithms that can learn from data.

ML programs are used for natural language processing, computer vision, speech recognition, and more.

Deep learning is a subset of machine learning that uses multi-layered neural networks."""

chunks = r_splitter.split_text(sample_text)
print(f"Split into {len(chunks)} chunks:\n")
for i, chunk in enumerate(chunks):
    print(f"[Chunk {i+1}] ({len(chunk)} chars): {chunk}")
    print()

In [ ]:
# Compare RecursiveCharacterTextSplitter vs CharacterTextSplitter
c_splitter = CharacterTextSplitter(chunk_size=150, chunk_overlap=20, separator=' ')
print(f"RecursiveCharacterTextSplitter : {len(r_splitter.split_text(sample_text))} chunks")
print(f"CharacterTextSplitter (space)  : {len(c_splitter.split_text(sample_text))} chunks")
print("\nThe recursive splitter tries paragraph, sentence, then word boundaries before splitting mid-word.")

In [ ]:
# Markdown-aware splitting preserves header hierarchy as chunk metadata
from langchain_text_splitters import MarkdownHeaderTextSplitter

markdown_doc = """# Machine Learning

## Supervised Learning
Uses labelled training data. The model learns a mapping from inputs to outputs.

## Unsupervised Learning
Finds patterns in unlabelled data, such as clustering or dimensionality reduction.

### Clustering
K-means and DBSCAN are popular clustering algorithms."""

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "H1"), ("##", "H2"), ("###", "H3")]
)
md_splits = md_splitter.split_text(markdown_doc)

for s in md_splits:
    print(f"Metadata: {s.metadata}")
    print(f"Content:  {s.page_content[:100]}")
    print()

In [ ]:
# Split all loaded documents into 1000-character chunks with 150-character overlap
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

web_splits = text_splitter.split_documents(all_docs)
pdf_splits = text_splitter.split_documents(pdf_pages)
all_splits = web_splits + pdf_splits

print(f"Web  : {len(all_docs)} documents -> {len(web_splits)} chunks")
print(f"PDF  : {len(pdf_pages)} pages -> {len(pdf_splits)} chunks")
print(f"Total: {len(all_splits)} chunks")

sample_chunk = all_splits[10]
print("\nSample chunk content:", sample_chunk.page_content[:300])
print("\nMetadata:", sample_chunk.metadata)

---
## Part 3: Vector Stores and Embeddings

Chunks are converted to dense vector representations using `all-MiniLM-L6-v2`, a lightweight HuggingFace model that runs locally on CPU with no API calls.

### 3a. Embedding similarity

In [ ]:
s1 = "machine learning is a subset of artificial intelligence"
s2 = "ML is part of the broader field of AI"
s3 = "the stock market closed higher on Friday"

e1 = embedding.embed_query(s1)
e2 = embedding.embed_query(s2)
e3 = embedding.embed_query(s3)

def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"Embedding dimensions: {len(e1)}")
print(f"\nSimilarity scores (1.0 = identical meaning):")
print(f"  s1 vs s2 (same topic, different wording): {cosine_sim(e1, e2):.4f}")
print(f"  s1 vs s3 (unrelated topic):               {cosine_sim(e1, e3):.4f}")

### 3b. Building the ChromaDB vector store

In [ ]:
persist_directory = '/content/chroma_db'  # /content is always writable in Colab
if os.path.exists(persist_directory):
    shutil.rmtree(persist_directory)

print("Embedding all chunks locally. This takes 1-2 minutes on Colab CPU...")

vectordb = Chroma.from_documents(
    documents=all_splits,
    embedding=embedding,
    persist_directory=persist_directory
)

print(f"Vector store ready. Vectors indexed: {vectordb._collection.count()}")

In [ ]:
question = "What is the difference between supervised and unsupervised learning?"
docs = vectordb.similarity_search(question, k=3)

print(f"Top {len(docs)} results:\n")
for i, doc in enumerate(docs):
    print(f"--- Result {i+1} (source: ...{str(doc.metadata.get('source','?'))[-50:]}) ---")
    print(doc.page_content[:300])
    print()

### 3c. Maximum Marginal Relevance (MMR)

Standard similarity search can return near-duplicate chunks. MMR balances relevance with diversity to return a broader set of results.

In [ ]:
question = "How do neural networks learn?"

docs_ss  = vectordb.similarity_search(question, k=3)
docs_mmr = vectordb.max_marginal_relevance_search(question, k=3, fetch_k=6)

print("--- Similarity Search ---")
for i, d in enumerate(docs_ss):
    print(f"[{i+1}] {d.page_content[:130]}")

print("\n--- MMR (diverse results) ---")
for i, d in enumerate(docs_mmr):
    print(f"[{i+1}] {d.page_content[:130]}")

---
## Part 4: Advanced Retrieval

### 4a. Metadata filtering

In [ ]:
question = "What are transformers used for?"

docs_all = vectordb.similarity_search(question, k=3)
print("Without filter:")
for d in docs_all:
    print(f"  source: ...{str(d.metadata.get('source','?'))[-55:]}")

print()
docs_pdf = vectordb.similarity_search(question, k=3, filter={"source": pdf_path})
print("PDF source only:")
for d in docs_pdf:
    print(f"  page {d.metadata.get('page','?')}: {d.page_content[:100]}")

### 4b. LLM-guided source routing

The LLM classifies each question and selects the appropriate source to search. This achieves the same goal as `SelfQueryRetriever` without the brittle dependency on query constructor parsers.

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

filter_prompt = PromptTemplate.from_template("""
You are a metadata filter extractor. Given a user question, decide which source to search.

Available sources:
- "wikipedia" (articles on Machine Learning, Deep Learning, NLP, Large Language Models)
- "pdf" (the Attention Is All You Need Transformer research paper)
- "all" (search everything)

User question: {question}

Reply with ONLY one word: wikipedia, pdf, or all.
""")

filter_chain = filter_prompt | llm | StrOutputParser()

def smart_retrieve(question, k=3):
    source_filter = safe_invoke(filter_chain, {"question": question}).strip().lower()
    print(f"  Source selected: '{source_filter}'")

    if "pdf" in source_filter:
        docs = vectordb.similarity_search(question, k=k, filter={"source": pdf_path})
        print("  Searching PDF only")
    elif "wikipedia" in source_filter:
        docs = vectordb.similarity_search(question, k=k+2)
        docs = [d for d in docs if 'wikipedia' in str(d.metadata.get('source', ''))][:k]
        print("  Searching Wikipedia only")
    else:
        docs = vectordb.similarity_search(question, k=k)
        print("  Searching all sources")

    return docs

questions = [
    "What does the transformer paper say about multi-head attention?",
    "What does Wikipedia say about supervised learning?",
    "What is backpropagation?",
]

for q in questions:
    print(f"\nQ: {q}")
    results = smart_retrieve(q, k=2)
    for d in results:
        src = str(d.metadata.get('source', '?'))[-45:]
        page = d.metadata.get('page', '-')
        print(f"  [p.{page} | ...{src}] {d.page_content[:120]}")
    time.sleep(4)

### 4c. Contextual Compression

Each retrieved chunk is passed through an LLM extractor that returns only the portion relevant to the query, reducing noise in the context sent to the final LLM.

In [ ]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

compression_retriever = ContextualCompressionRetriever(
    base_compressor=LLMChainExtractor.from_llm(llm),
    base_retriever=vectordb.as_retriever(search_type="mmr")
)

compressed_docs = compression_retriever.invoke(
    "What are the main types of machine learning?"
)

print(f"Retrieved {len(compressed_docs)} compressed chunk(s):\n")
for i, doc in enumerate(compressed_docs):
    print(f"--- Doc {i+1} ---\n{doc.page_content[:400]}\n")

---
## Part 5: Question Answering

### 5a. Basic RetrievalQA chain

In [ ]:
from langchain_classic.chains.retrieval_qa.base import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(llm, retriever=vectordb.as_retriever())
result = safe_invoke(qa_chain, {"query": "What is the attention mechanism in the Transformer model?"})
print(result["result"])

### 5b. Custom prompt template

In [ ]:
from langchain_core.prompts import PromptTemplate

template = """You are a helpful assistant. Answer using only the context below.
If the answer is not in the context, say so. Keep answers to 3 sentences maximum.

Context:
{context}

Question: {question}
Answer:"""

qa_chain_custom = RetrievalQA.from_chain_type(
    llm,
    retriever=vectordb.as_retriever(),
    return_source_documents=True,
    chain_type_kwargs={"prompt": PromptTemplate.from_template(template)}
)

result = qa_chain_custom.invoke({"query": "How does backpropagation work in neural networks?"})
print("Answer:", result["result"])
print("\nSource documents:")
for doc in result["source_documents"]:
    print(f"  ...{str(doc.metadata.get('source','?'))[-60:]}")

### 5c. Chain types

| Chain | Behaviour |
|---|---|
| `stuff` | All retrieved chunks placed in a single prompt (fast, default) |
| `map_reduce` | Each chunk summarised individually, then combined into a final answer |
| `refine` | Answer iteratively refined as each chunk is processed |

In [ ]:
question = "What are the applications of large language models?"

for chain_type in ["stuff", "map_reduce"]:
    qa = RetrievalQA.from_chain_type(llm, retriever=vectordb.as_retriever(), chain_type=chain_type)
    result = safe_invoke(qa, {"query": question})
    print(f"--- {chain_type.upper()} ---")
    print(result["result"])
    print()
    time.sleep(4)

### 5d. The stateless problem

`RetrievalQA` does not retain conversation history. Each call is independent, so follow-up questions that reference earlier answers will fail.

In [ ]:
qa = RetrievalQA.from_chain_type(llm, retriever=vectordb.as_retriever())

r1 = safe_invoke(qa, {"query": "What is supervised learning?"})
print("Q1: What is supervised learning?\nA1:", r1["result"])

time.sleep(4)

r2 = safe_invoke(qa, {"query": "Can you give me an example of that?"})
print("\nQ2: Can you give me an example of that?\nA2:", r2["result"])
print("\nNote: 'that' has no referent -- the chain has no memory of Q1. Addressed in Part 6.")

---
## Part 6: Conversational Chat with Memory

`ConversationalRetrievalChain` stores the conversation history and rephrases follow-up questions in context before passing them to the retriever.

### 6a. Basic conversational chain

In [ ]:
from langchain_classic.chains.conversational_retrieval.base import ConversationalRetrievalChain
from langchain_classic.memory.buffer import ConversationBufferMemory

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

qa_conv = ConversationalRetrievalChain.from_llm(
    llm,
    retriever=vectordb.as_retriever(),
    memory=memory
)

for q in [
    "What is supervised learning?",
    "Can you give me a concrete example of that?",
    "How does that compare to unsupervised learning?"
]:
    result = safe_invoke(qa_conv, {"question": q})
    print(f"Q: {q}\nA: {result['answer']}\n")
    time.sleep(4)

In [ ]:
print("Conversation history:")
for msg in memory.chat_memory.messages:
    role = "User" if msg.type == "human" else "Assistant"
    print(f"  [{role}]: {str(msg.content)[:100]}")

### 6b. Chatbot with custom prompt and MMR retrieval

In [ ]:
def create_chatbot(vectordb, llm):
    memory = ConversationBufferMemory(
        memory_key="chat_history", return_messages=True, output_key='answer'
    )
    prompt = PromptTemplate.from_template(
        """You are a knowledgeable assistant. Answer using the context provided.
If the answer is not in the context, say so. Be concise.

Context: {context}
Question: {question}
Answer:"""
    )
    return ConversationalRetrievalChain.from_llm(
        llm,
        retriever=vectordb.as_retriever(search_type="mmr", search_kwargs={"k": 4}),
        memory=memory,
        return_source_documents=True,
        combine_docs_chain_kwargs={"prompt": prompt}
    )

def chat(chain, question):
    result = chain.invoke({"question": question})
    print(f"\nUser: {question}")
    print(f"\nAssistant: {result['answer']}")
    sources = set(str(d.metadata.get('source',''))[-50:] for d in result.get('source_documents', []))
    if sources:
        print(f"\nSources: {', '.join(sources)}")
    print("-" * 60)

print("Helper functions defined.")

In [ ]:
chatbot = create_chatbot(vectordb, llm)
print("Chatbot ready. Knowledge base covers: machine learning, deep learning, NLP, Transformers.")

In [ ]:
chat(chatbot, "What is deep learning and how is it different from traditional machine learning?")
time.sleep(4)

In [ ]:
chat(chatbot, "What kinds of architectures are commonly used in it?")
time.sleep(4)

In [ ]:
chat(chatbot, "How do Transformers improve on those architectures?")
time.sleep(4)

In [ ]:
chat(chatbot, "What are large language models and how do they relate to Transformers?")
time.sleep(4)

In [ ]:
chat(chatbot, "What are some real-world applications of NLP?")
time.sleep(4)

### 6c. Chat with an uploaded PDF

In [ ]:
from google.colab import files
from langchain_community.vectorstores import DocArrayInMemorySearch

def chat_with_your_pdf():
    print("Upload a PDF:")
    uploaded = files.upload()
    if not uploaded:
        return None

    filename = list(uploaded.keys())[0]
    loader = PyPDFLoader(filename)
    splits = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150).split_documents(loader.load())
    print(f"{filename} loaded and split into {len(splits)} chunks.")

    db = DocArrayInMemorySearch.from_documents(splits, embedding)
    memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True, output_key='answer')
    chain = ConversationalRetrievalChain.from_llm(
        llm, retriever=db.as_retriever(search_kwargs={"k": 4}),
        memory=memory, return_source_documents=True
    )
    print(f"Ready. Ask questions about '{filename}'.")
    return chain

my_chatbot = chat_with_your_pdf()
chat(my_chatbot, "What is this document about?")

---
## Part 7: Alternative Stack — FAISS and Nomic Embeddings

This section demonstrates an alternative retrieval pipeline using FAISS (an in-memory vector store) and the `nomic-embed-text-v1.5` embedding model, which offers higher retrieval accuracy than `all-MiniLM-L6-v2`. The same Groq LLM from the setup cell is reused.

| Component | Parts 1-6 | Part 7 |
|-----------|-----------|--------|
| Vector Store | ChromaDB (persistent) | FAISS (in-memory) |
| Embeddings | all-MiniLM-L6-v2 | nomic-embed-text-v1.5 |
| LLM | Groq | Groq (same instance) |
| Source | Wikipedia + arXiv PDF | Uploaded PDF |

In [ ]:
# The Groq LLM initialised in the setup cell is reused here.
print("Using existing LLM:", GROQ_MODEL)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings as HFEmbeddings
from langchain_community.vectorstores import FAISS

# nomic-embed-text-v1.5 produces higher quality embeddings than all-MiniLM-L6-v2
nomic_embedding = HFEmbeddings(
    model_name="nomic-ai/nomic-embed-text-v1.5",
    model_kwargs={"trust_remote_code": True}
)

print("Nomic embedding model ready.")

### 7a. Upload and index a PDF

In [ ]:
from google.colab import files as colab_files

print("Upload a PDF:")
uploaded = colab_files.upload()

if uploaded:
    pdf_filename = list(uploaded.keys())[0]
    print(f"Uploaded: {pdf_filename}")

    user_loader = PyPDFLoader(pdf_filename)
    user_documents = user_loader.load()
    print(f"Pages loaded: {len(user_documents)}")

    user_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
    user_texts = user_splitter.split_documents(user_documents)

    for idx, t in enumerate(user_texts):
        t.metadata["id"] = idx

    print(f"Chunks created: {len(user_texts)}")
    print("\nSample chunk:")
    print(user_texts[0].page_content[:300])
else:
    print("No file uploaded.")

### 7b. Build FAISS index and run Q&A

In [ ]:
if uploaded:
    print("Building FAISS index with Nomic embeddings...")
    faiss_retriever = FAISS.from_documents(user_texts, nomic_embedding).as_retriever(
        search_kwargs={"k": 5}
    )

    def pretty_print_docs(docs):
        for i, d in enumerate(docs):
            print(f"{'─' * 50}\nDocument {i+1}:")
            print(f"Content:\n{d.page_content}\n")
            for key, value in d.metadata.items():
                print(f"  {key}: {value}")
        print("─" * 50)

    groq_qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=faiss_retriever)

    query = "Give me a summary of this document."
    response = groq_qa_chain.invoke(query)
    print("\nAnswer:")
    print(response["result"])
else:
    print("Upload a PDF in the cell above first.")

In [ ]:
if uploaded:
    my_query = "What are the main conclusions of this document?"  # change this line
    response = groq_qa_chain.invoke(my_query)
    print(f"Q: {my_query}\n")
    print("A:", response["result"])